In [1]:
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import numpy as np
import random
import hashlib
import json
import sys
import platform
import sklearn

In [2]:
PROJECT_DIR = Path(r"C:\Users\Joseph (YO)\JGM\U\Semestre_10\InteligenciaArtificial\Proyecto_Yuca")

REPORTS_DIR = PROJECT_DIR / "reports"
CONFIGS_DIR = PROJECT_DIR / "configs"

CONFIGS_DIR.mkdir(parents=True, exist_ok=True)

SPLITS_PATH = REPORTS_DIR / "dataset_splits.csv"
MAPPING_PATH = REPORTS_DIR / "class_mapping.json"
MANIFEST_PATH = REPORTS_DIR / "images_manifest.csv"

df_splits = pd.read_csv(SPLITS_PATH)

with open(MAPPING_PATH, "r", encoding="utf-8") as f:
    class_mapping = json.load(f)

print("Imágenes registradas:", len(df_splits))
print("Mapeo de clases:", class_mapping)

display(df_splits.head())

Imágenes registradas: 21397
Mapeo de clases: {'bacterial_blight': 0, 'brown_streak_disease': 1, 'green_mottle': 2, 'healthy': 3, 'mosaic_disease': 4}


,filename,filepath,class,extension,width,height,format,mode,size_bytes,sha256,status,error,split,fold
0,2422491518.jpg,mosaic_disease\2422491518.jpg,mosaic_disease,.jpg,800,600,JPEG,RGB,124540,568dbafb87aec5908d558598dd7391d0bd6c15e7fd33fd...,valid,NaN,development,3
1,2878596544.jpg,mosaic_disease\2878596544.jpg,mosaic_disease,.jpg,800,600,JPEG,RGB,143096,a24b7bc16ac427258d93de3bc06c78625e5eb632ba8f93...,valid,NaN,development,2
2,1696364421.jpg,green_mottle\1696364421.jpg,green_mottle,.jpg,800,600,JPEG,RGB,82996,5037e2d79f82f25519924a581be8cac219bef654cc8957...,valid,NaN,development,3
3,3806158675.jpg,mosaic_disease\3806158675.jpg,mosaic_disease,.jpg,800,600,JPEG,RGB,127459,25599e588ba4e49fd8bb6874db71e42da827bee10ba49e...,valid,NaN,development,3
4,1132695066.jpg,mosaic_disease\1132695066.jpg,mosaic_disease,.jpg,800,600,JPEG,RGB,110081,000e1d7184525f915868312102bc8f422fdd586594473c...,valid,NaN,development,2


In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Semilla configurada:", SEED)

Semilla configurada: 42


In [4]:
distribution_splits = pd.crosstab(
    df_splits["class"],
    df_splits["split"]
)

distribution_splits["total"] = (
    distribution_splits.sum(axis=1)
)

display(distribution_splits)

distribution_splits.to_csv(
    REPORTS_DIR / "split_distribution.csv",
    encoding="utf-8-sig"
)

split,development,test,total
class,,,
bacterial_blight,924,163,1087
brown_streak_disease,1861,328,2189
green_mottle,2028,358,2386
healthy,2190,387,2577
mosaic_disease,11184,1974,13158


In [5]:
distribution_percentages = (
    pd.crosstab(
        df_splits["class"],
        df_splits["split"],
        normalize="index"
    ) * 100
).round(2)

display(distribution_percentages)

distribution_percentages.to_csv(
    REPORTS_DIR / "split_distribution_percentages.csv",
    encoding="utf-8-sig"
)

split,development,test
class,,
bacterial_blight,85.00,15.00
brown_streak_disease,85.02,14.98
green_mottle,85.00,15.00
healthy,84.98,15.02
mosaic_disease,85.00,15.00


In [6]:
df_development = df_splits[
    df_splits["split"] == "development"
].copy()

fold_distribution = pd.crosstab(
    df_development["class"],
    df_development["fold"]
)

fold_distribution["total"] = (
    fold_distribution.sum(axis=1)
)

display(fold_distribution)

fold_distribution.to_csv(
    REPORTS_DIR / "fold_distribution.csv",
    encoding="utf-8-sig"
)

fold,1,2,3,total
class,,,,
bacterial_blight,308,308,308,924
brown_streak_disease,621,620,620,1861
green_mottle,676,676,676,2028
healthy,730,730,730,2190
mosaic_disease,3728,3728,3728,11184


In [7]:
df_test = df_splits[
    df_splits["split"] == "test"
].copy()

# Verificar que las particiones sean las esperadas
assert set(df_splits["split"].unique()) == {
    "development", "test"
}

# Verificar que no existan rutas repetidas
assert df_splits["filepath"].is_unique, (
    "Existen rutas duplicadas."
)

# Verificar que todas las imágenes sean válidas
assert df_splits["status"].eq("valid").all(), (
    "Existen imágenes inválidas."
)

# Verificar que todos los hashes estén disponibles
assert df_splits["sha256"].notna().all(), (
    "Existen imágenes sin hash."
)

# Verificar que no haya hashes compartidos
# entre desarrollo y prueba
development_hashes = set(df_development["sha256"])
test_hashes = set(df_test["sha256"])

assert development_hashes.isdisjoint(test_hashes), (
    "Existen imágenes idénticas entre desarrollo y prueba."
)

# Verificar que los folds sean correctos
assert set(df_development["fold"].unique()) == {1, 2, 3}

# Verificar que test no participe en los folds
assert df_test["fold"].eq(-1).all()

# Verificar que las clases coincidan con el mapeo
assert set(df_splits["class"].unique()) == set(class_mapping)

# Verificar que no haya hashes compartidos entre folds
assert (
    df_development.groupby("sha256")["fold"]
    .nunique()
    .le(1)
    .all()
), "Existen imágenes idénticas en diferentes folds."

print("Todas las verificaciones fueron superadas.")

Todas las verificaciones fueron superadas.


In [8]:
dataset_config = {
    "dataset_name": "Cassava Leaf Disease",

    "dataset_version": "1.0",

    "task": "multiclass_image_classification",

    "seed": SEED,

    "partition": {
        "development_size": 0.85,
        "test_size": 0.15,
        "stratified": True
    },

    "cross_validation": {
        "method": "StratifiedKFold",
        "n_splits": 3,
        "shuffle": True,
        "random_state": SEED
    },

    "dataset_statistics": {
        "total_images": len(df_splits),
        "development_images": len(df_development),
        "test_images": len(df_test),
        "num_classes": len(class_mapping)
    },

    "class_mapping": class_mapping,

    "files": {
        "manifest": "reports/images_manifest.csv",
        "splits": "reports/dataset_splits.csv",
        "class_mapping": "reports/class_mapping.json"
    },

    "created_at": datetime.now(timezone.utc).isoformat()
}

config_path = CONFIGS_DIR / "dataset_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        dataset_config,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Configuración guardada en:", config_path)

Configuración guardada en: C:\Users\Joseph (YO)\JGM\U\Semestre_10\InteligenciaArtificial\Proyecto_Yuca\configs\dataset_config.json


In [9]:
def calculate_sha256(filepath):

    sha256 = hashlib.sha256()

    with open(filepath, "rb") as f:

        for block in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            sha256.update(block)

    return sha256.hexdigest()


files_to_verify = [
    MANIFEST_PATH,
    SPLITS_PATH,
    MAPPING_PATH,
    config_path
]

integrity_records = []

for filepath in files_to_verify:

    integrity_records.append({
        "file": str(filepath.relative_to(PROJECT_DIR)),
        "sha256": calculate_sha256(filepath)
    })

df_integrity = pd.DataFrame(integrity_records)

display(df_integrity)

df_integrity.to_csv(
    REPORTS_DIR / "file_integrity.csv",
    index=False,
    encoding="utf-8-sig"
)

,file,sha256
0,reports\images_manifest.csv,b83c4a3594f488ac2a3e6a9696491cfe03c24baf248d41...
1,reports\dataset_splits.csv,21d5f7c006c38fecead271b56c0bf68e4938cdde24deee...
2,reports\class_mapping.json,b49e6c123ee29bfe1c092856accea64e4024a1cb63da6b...
3,configs\dataset_config.json,ffd81b9e384698d9854e239621969203104401ce808090...


In [10]:
saved_integrity = pd.read_csv(
    REPORTS_DIR / "file_integrity.csv"
)

for _, row in saved_integrity.iterrows():

    filepath = PROJECT_DIR / row["file"]

    current_hash = calculate_sha256(filepath)

    assert current_hash == row["sha256"], (
        f"El archivo fue modificado: {row['file']}"
    )

    print(f"Integridad correcta: {row['file']}")

Integridad correcta: reports\images_manifest.csv
Integridad correcta: reports\dataset_splits.csv
Integridad correcta: reports\class_mapping.json
Integridad correcta: configs\dataset_config.json


In [11]:
import PIL

environment_info = {
    "python_version": sys.version,
    "operating_system": platform.system(),
    "platform": platform.platform(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "pillow_version": PIL.__version__,
    "scikit_learn_version": sklearn.__version__
}

with open(
    CONFIGS_DIR / "environment_info.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        environment_info,
        f,
        indent=4
    )

display(environment_info)

{'python_version': '3.11.0 (main, Oct 24 2022, 18:26:48) [MSC v.1933 64 bit (AMD64)]',
 'operating_system': 'Windows',
 'platform': 'Windows-10-10.0.19045-SP0',
 'pandas_version': '3.1.1',
 'numpy_version': '2.4.6',
 'pillow_version': '13.0.0',
 'scikit_learn_version': '1.9.1'}